<a href="https://colab.research.google.com/github/Shashith240/Statistical-Learning-e22240/blob/main/Assignment_7c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



## Part 1: Q. Bayesian Estimation of a User Ability Parameter from Item Responses

### 1. Visualizing the Mechanics

The probability of a correct response is modeled by the 2PL function:


$$P(Y_i = 1 \mid \Theta = \theta) = \frac{1}{1 + e^{-a_i(\theta - b_i)}}$$

* **Interpretation of $b_i$:** The difficulty parameter $b_i$ acts as a horizontal shift parameter. When $\theta = b_i$, the response probability is exactly $0.5$. Increasing $b_i$ shifts the entire logistic curve to the right, meaning a user needs a higher latent ability $\theta$ to achieve the same probability of success. Conversely, decreasing $b_i$ shifts the curve to the left (easier item).

### 2. Sequential Likelihood Contribution

For a single isolated response $y_k \in \{0, 1\}$ at step $k$:


$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

For the running history vector $y^{(k)} = (y_1, y_2, \dots, y_k)$, assuming conditional independence given $\theta$:


$$L(y^{(k)} \mid \theta) = \prod_{i=1}^k [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

### 3. Mathematical Formulation of the Running Update

Using Bayes' Theorem sequentially, the posterior at step $k$ is proportional to the product of the likelihood contribution of the new observation $y_k$ and the prior state (which is the posterior from step $k-1$):


$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$$

$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto \left([p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}\right) f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$$

### 4. Dynamic Shifting

When a user correctly answers ($y_k = 1$) a highly difficult item (large $b_k$), the likelihood function $p_k(\theta)$ is a logistic curve that remains very close to 0 for lower values of $\theta$ and climbs toward 1 only at higher values of $\theta$. Multiplying the previous posterior $f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$ by this heavily right-skewed likelihood profile suppresses the density values for lower $\theta$ ranges and amplifies them for higher $\theta$ ranges. Consequently, upon normalization, the mode (peak) of the running posterior density shifts significantly to the right.

### 5. Tracking Certainty and Sharpness

The discrimination parameter $a_k$ governs the steepness of the logistic curve.

* **Large $a_k$:** The item response function behaves like a sharp step function. Getting it right or wrong provides highly distinct evidence, resulting in a narrow, high-peaked likelihood contribution. This significantly reduces the variance of the updated posterior, increasing its **sharpness** (certainty).
* **Small $a_k$:** The item response curve is very flat, meaning users of high or low ability have similar probabilities of success. The likelihood function provides very little new information, causing the posterior variance to remain largely unchanged.

### 6. Numerical Implementation of a Running Grid

Because the 2PL likelihood lacks a conjugate prior pairing, updates must be computed numerically over a fixed grid:

1. Define a dense linear grid vector $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$ over a reasonable domain (e.g., $[-5, 5]$).
2. Initialize an array representing the prior evaluation: $\mathbf{f}^{(0)} = [f_1^{(0)}, \dots, f_M^{(0)}]$ where $f_j^{(0)} = \frac{1}{\sqrt{2\pi}}e^{-\theta_j^2/2}$.
3. For each step $k$:
* Evaluate the likelihood vector $\mathbf{L}_k = [L(y_k \mid \theta_1), \dots, L(y_k \mid \theta_M)]$.
* Perform an element-wise multiplication to get the unnormalized posterior: $\mathbf{\tilde{f}}^{(k)} = \mathbf{L}_k \odot \mathbf{f}^{(k-1)}$.
* Compute the total area under the curve using numerical integration (e.g., the Trapezoidal Rule): $I = \sum_{j=1}^{M-1} \frac{\tilde{f}_j^{(k)} + \tilde{f}_{j+1}^{(k)}}{2} \Delta\theta$.
* Divide by the normalizing constant: $\mathbf{f}^{(k)} = \frac{\mathbf{\tilde{f}}^{(k)}}{I}$.



### 7. Python Implementation & Convergence Tracking

```python
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Setup Simulation Parameters
np.random.seed(42)
theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-5, 5, 1000)

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Generate Random Item Parameters
a_params = np.random.uniform(0.5, 2.0, size=n_items)
b_params = np.random.normal(0, 1, size=n_items)

running_bayes = [0.0]
running_map = [0.0]
steps = list(range(n_items + 1))

# Initialize prior distribution: N(0, 1)
current_posterior = stats.norm.pdf(theta_grid, 0, 1)

for k in range(n_items):
    a_k, b_k = a_params[k], b_params[k]
    
    # Stochastic response generation
    prob_true = p_i(theta_true, a_k, b_k)
    y_k = 1 if np.random.uniform(0, 1) < prob_true else 0
    
    # Grid update
    prob_grid = p_i(theta_grid, a_k, b_k)
    likelihood = (prob_grid ** y_k) * ((1 - prob_grid) ** (1 - y_k))
    current_posterior *= likelihood
    
    # Normalize
    integral = np.trapezoid(current_posterior, theta_grid)
    current_posterior /= integral
    
    # Point Estimates
    theta_bayes_k = np.trapezoid(theta_grid * current_posterior, theta_grid)
    theta_map_k = theta_grid[np.argmax(current_posterior)]
    
    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

# Visualization
fig = go.Figure()
fig.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True Ability (0.75)")
fig.add_trace(go.Scatter(x=steps, y=running_bayes, mode='lines+markers', name='Posterior Mean (Bayes)'))
fig.add_trace(go.Scatter(x=steps, y=running_map, mode='lines+markers', name='MAP Estimate'))
fig.update_layout(title="Convergence of Latent Ability Over Time", xaxis_title="Item Number (k)", yaxis_title="Estimate")
fig.show()

```

---

## Part 2: Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

### 1. Structural Probability and Properties

The Beta distribution PDF is expressed as:


$$f(\theta \mid \alpha, \beta) = \frac{1}{\mathrm{B}(\alpha, \beta)} \theta^{\alpha - 1} (1 - \theta)^{\beta - 1}$$

* **Interpretation:** The parameters $\alpha$ and $\beta$ act as pseudo-counts for conversions (clicks) and non-conversions (non-clicks).
* When $\alpha = \beta = 1$, the distribution is flat ($\mathrm{Uniform}(0,1)$), representing absolute uncertainty.
* When $\alpha < \beta$ (e.g., $2, 8$), the center of mass shifts left, reflecting a low expected conversion rate.
* When $\alpha > \beta$ (e.g., $8, 2$), the center of mass shifts right, favoring high conversion values.



### 2. Sequential Likelihood and Joint History

For a single impression outcome $y_k \in \{0, 1\}$:


$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

For the running sequence history $y^{(k)}$:


$$L(y^{(k)} \mid \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum y_i} (1 - \theta)^{k - \sum y_i}$$

### 3. Closed-Form Analytical Updates (Conjugacy)

Given a prior at step $k-1$ belonging to the Beta family, $f(\theta) \propto \theta^{\alpha_{k-1}-1}(1-\theta)^{\beta_{k-1}-1}$:


$$f(\theta \mid y^{(k)}) \propto L(y_k \mid \theta) \cdot f(\theta \mid y^{(k-1)})$$

$$f(\theta \mid y^{(k)}) \propto \left[\theta^{y_k} (1 - \theta)^{1 - y_k}\right] \cdot \left[\theta^{\alpha_{k-1}-1} (1 - \theta)^{\beta_{k-1}-1}\right]$$

$$f(\theta \mid y^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

This structurally matches the kernel of a Beta distribution, proving **conjugacy**. The exact arithmetic updates are:


$$\alpha_k = \alpha_{k-1} + y_k$$

$$\beta_k = \beta_{k-1} + (1 - y_k)$$

### 4. Dynamic Shifting Mechanics

* If a user clicks ($y_k = 1$), $\alpha_k$ increments while $\beta_k$ stays constant, pulling the distribution's mode toward 1.
* If a user passes ($y_k = 0$), $\beta_k$ increments, pulling the peak towards 0.
* **Contrast:** Unlike the 2PL IRT model which forces computationally heavy numerical integration over grids at every incoming sample, the conjugate Beta-Binomial setup evaluates updates instantly with basic addition ($+1$ or $+0$).

### 5. Running Point Estimators

Directly derived from the shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean ($\hat{\theta}^{(k)}_{\mathrm{Bayes}}$):**

$$\mathbb{E}[\Theta \mid y^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$


* **Running Maximum A Posteriori ($\hat{\theta}^{(k)}_{\mathrm{MAP}}$):**

$$\hat{\theta}^{(k)}_{\mathrm{MAP}} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2} \quad (\text{for } \alpha_k, \beta_k > 1)$$



### 6. Performance Tracking and Convergence Analysis

```python
import numpy as np
import plotly.graph_objects as go

# Initialize Simulation Parameters
np.random.seed(42)
theta_true = 0.35
n_impressions = 100

# Base Prior: Uniform State
alpha_k = 1
beta_k = 1

running_bayes = [alpha_k / (alpha_k + beta_k)]
running_map = [0.0]  # Undefined mode for Uniform, initialize at 0
steps = list(range(n_impressions + 1))

for k in range(n_impressions):
    # Simulate Bernoulli User Response
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0
    
    # Exact Closed-Form Update
    alpha_k += y_k
    beta_k += (1 - y_k)
    
    # Calculate exact point estimators
    bayes_est = alpha_k / (alpha_k + beta_k)
    map_est = (alpha_k - 1) / (alpha_k + beta_k - 2) if (alpha_k + beta_k > 2) else 0.5
    
    running_bayes.append(bayes_est)
    running_map.append(map_est)

# Visualize tracking performance
fig2 = go.Figure()
fig2.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True CTR (0.35)")
fig2.add_trace(go.Scatter(x=steps, y=running_bayes, mode='lines', name='Posterior Mean'))
fig2.add_trace(go.Scatter(x=steps, y=running_map, mode='lines', name='MAP Estimate'))
fig2.update_layout(title="Beta-Binomial Sequential CTR Tracking", xaxis_title="Impressions (k)", yaxis_title="CTR Estimate")
fig2.show()

```